# 01 – Camada Gold

No repositório [Literacy Data Pipeline](https://github.com/paollalopes/literacy-data-pipeline) a pipeline foi construída em **Databricks + Apache Spark**, lendo e gravando no **Amazon S3**(arquitetura Medalhão: Bronze → Silver → Gold).

Nesta fase, o projeto passa a rodar em **notebooks locais e versionados no GitHub**. Para que a modelagem seja **reproduzível** por qualquer pessoa que clonar o repositório, as regras de negócio da Silver e da Gold foram reimplementadas em **pandas** no módulo `src/preprocessing/gold.py`, partindo dos mesmos seis CSVs originais (Bronze).

**Objetivos deste notebook**
1. Ler a Bronze (CSVs originais).
2. Aplicar os tratamentos da Silver e as regras de qualidade.
3. Gerar as tabelas Gold em Parquet (`data/processed/`).
4. Conferir que os resultados batem com os números da Fase 2.

In [6]:
# Configura o caminho do projeto para importar os módulos de src/.
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

## 1. Bronze – Leitura dos arquivos originais

In [8]:
# Lê os seis CSVs originais sem nenhuma alteração.
from src.preprocessing.gold import carregar_bronze, construir_silver, validar_silver, construir_gold

bronze = carregar_bronze()
pd.DataFrame(
    [(nome, df.shape[0], df.shape[1]) for nome, df in bronze.items()],
    columns=["dataset", "registros", "colunas"],
)

,dataset,registros,colunas
0,avaliacao_municipio,23995,15
1,avaliacao_uf,145,15
2,avaliacao_alunos,103958,12
3,meta_brasil,3,11
4,meta_municipio,10704,13
5,meta_uf,54,12


## 2. Silver – Padronização e qualidade

Tratamentos aplicados:
- remoção de duplicidades exatas;
- padronização de tipos e de siglas de UF;
- descrição da rede de ensino;
- derivação de **UF** e **região** a partir do código IBGE do município (os dois primeiros dígitos identificam a UF), permitindo análises territoriais na base de alunos e de municípios.

> **Correção em relação à Fase 2:** o código de rede `5` havia sido rotulado como *Privada*.
> A análise abaixo mostra que ele é a **rede Pública** (consolidação de Estadual + Municipal):
> quando o município só tem rede Municipal, a taxa da rede 5 é idêntica à Municipal; quando tem as duas,
> fica sempre entre elas.

In [9]:
# Evidência de que rede 5 = Pública: compara as taxas das redes 2, 3 e 5 para alguns municípios.
aval = bronze["avaliacao_municipio"]
comparacao = aval.pivot_table(index=["ano", "id_municipio"], columns="rede", values="taxa_alfabetizacao")
comparacao.head(12)

rede                0      2      3      5
ano  id_municipio                         
2023 1100015      NaN    NaN  64.55  64.55
     1100023      NaN    NaN  62.30  62.30
     1100031      NaN    NaN  69.10  69.10
     1100049      NaN  66.13  62.51  63.37
     1100056      NaN    NaN  58.53  58.53
     1100064      NaN    NaN  62.85  62.85
     1100072      NaN    NaN  58.20  58.20
     1100080      NaN    NaN  80.68  80.68
     1100098      NaN  69.18  85.33  81.65
     1100106      NaN  57.50  41.76  44.25
     1100114      NaN  61.54  62.82  62.75
     1100122      NaN  57.02  81.83  70.52

In [10]:
# Quantifica: em municípios com as redes 2, 3 e 5, a taxa da rede 5 fica entre as das redes 2 e 3?
c = comparacao.dropna(subset=[2, 3, 5])
entre = ((c[5] >= c[[2, 3]].min(axis=1) - 0.01) & (c[5] <= c[[2, 3]].max(axis=1) + 0.01)).mean()
so_municipal = comparacao[comparacao[2].isna()].dropna(subset=[3, 5])
iguais = (so_municipal[3] - so_municipal[5]).abs().lt(0.01).mean()
print(f"Rede 5 entre Estadual e Municipal: {entre:.1%} dos municípios com as três redes")
print(f"Rede 5 igual à Municipal quando só existe rede Municipal: {iguais:.1%}")

Rede 5 entre Estadual e Municipal: 100.0% dos municípios com as três redes
Rede 5 igual à Municipal quando só existe rede Municipal: 100.0%


In [11]:
# Constrói a Silver e executa as regras de qualidade.
silver = construir_silver(bronze)
validar_silver(silver)

,regra,ok,status
0,alunos: id_aluno+ano único,True,PASS
1,"alunos: binários em {0,1}",True,PASS
2,alunos: caderno não preenchido => sem proficiê...,True,PASS
3,alunos: alfabetizado == (proficiência >= 743),True,PASS
4,alunos: caderno não preenchido => alfabetizado...,True,PASS
5,"município: taxa em [0,100]",True,PASS
6,"metas município: metas em [0,100]",True,PASS
7,município: ano+município+rede único,True,PASS
8,metas município: ano+município único,True,PASS


### Achado importante para a modelagem

As regras acima confirmam dois comportamentos da base de alunos que serão decisivos nesta fase:

1. **`alfabetizado` é uma função direta de `proficiencia`** (proficiência ≥ 743 pontos, ponto de corte
   do INEP). Usar a proficiência como variável explicativa seria *data leakage* – o modelo “leria a resposta”.
2. **Alunos que não preencheram o caderno aparecem com `alfabetizado = 0`.** Para eles não sabemos se
   estão ou não alfabetizados; o rótulo é ausência de avaliação, não resultado.

In [12]:
# Distribuição do rótulo conforme o preenchimento do caderno.
alunos = silver["avaliacao_alunos"]
pd.crosstab(alunos["preenchimento_caderno"], alunos["alfabetizado"], margins=True)

alfabetizado,0,1,All
preenchimento_caderno,,,
0,13749,0,13749
1,36883,53326,90209
All,50632,53326,103958


## 3. Gold – Indicadores analíticos

In [13]:
# Gera e salva as tabelas Gold em data/processed/ (Parquet).
gold = construir_gold(salvar=True)
pd.DataFrame([(k, *v.shape) for k, v in gold.items()], columns=["tabela", "linhas", "colunas"])

,tabela,linhas,colunas
0,gold_brasil_desempenho,3,14
1,gold_uf_desempenho,145,7
2,gold_municipio_desempenho,10896,18
3,gold_municipio_evolucao,5500,10
4,gold_municipio_redes,11030,8
5,gold_alunos,103958,16
6,gold_indicadores_alunos,4,7


| Tabela | Conteúdo |
|---|---|
| `gold_brasil_desempenho` | Resultado nacional, metas, diferença para a meta e status |
| `gold_uf_desempenho` | Taxa e média de Português por UF e rede |
| `gold_municipio_desempenho` | Resultado da rede Municipal + metas municipais e status da meta |
| `gold_municipio_evolucao` | Evolução 2023 → 2024 por município |
| `gold_municipio_redes` | *(nova)* Taxa e média por rede (Estadual, Municipal, Pública) em formato largo |
| `gold_alunos` | *(nova)* Base de alunos tratada, com UF, região e flag de avaliação válida |
| `gold_indicadores_alunos` | Presença, avaliações válidas, alfabetização e proficiência por ano/rede |

In [14]:
# Brasil: resultado x meta.
gold["gold_brasil_desempenho"][["ano", "taxa_alfabetizacao", "meta_ano", "diferenca_para_meta",
                                "status_meta", "percentual_participacao"]]

,ano,taxa_alfabetizacao,meta_ano,diferenca_para_meta,status_meta,percentual_participacao
0,2023,55.9,NaN,NaN,SEM_META,86.00
1,2024,59.2,59.9,-0.7,NAO_ATINGIDA,87.37
2,2025,66.0,64.0,2.0,ATINGIDA,88.00


In [15]:
# Municípios com meta em 2024 e situação.
gm = gold["gold_municipio_desempenho"]
gm24 = gm[(gm["ano"] == 2024) & (gm["status_meta"] != "SEM_META")]
print("Municípios com meta em 2024:", len(gm24))
print(gm24["status_meta"].value_counts().to_string())
print(f"Taxa média observada: {gm24['taxa_alfabetizacao'].mean():.2f}%")
print(f"Meta média:           {gm24['meta_ano'].mean():.2f}%")

Municípios com meta em 2024: 5232
status_meta
ATINGIDA        2788
NAO_ATINGIDA    2444
Taxa média observada: 63.27%
Meta média:           62.15%


Os valores coincidem com a análise exploratória anterior (**5.232 municípios com meta, 2.788 atingiram, 2.444 não
atingiram, taxa média 63,27% vs. meta média 62,15%**), o que valida a reimplementação local.

In [16]:
# Evolução dos municípios entre 2023 e 2024.
gold["gold_municipio_evolucao"]["tendencia"].value_counts()

tendencia
MELHORA           3134
PIORA             2250
SEM_COMPARACAO     104
ESTAVEL             12
Name: count, dtype: int64

In [17]:
gold["gold_indicadores_alunos"]

,ano,rede_ensino,alunos,taxa_presenca,avaliacoes_validas,taxa_alfabetizacao_valida,proficiencia_media
0,2023,Estadual,4201,0.843133,3542,0.611519,748.882365
1,2023,Municipal,42946,0.862176,37016,0.579992,747.475851
2,2024,Estadual,7404,0.868855,6429,0.627936,751.985974
3,2024,Municipal,49407,0.875281,43222,0.593540,748.602425
